<a href="https://colab.research.google.com/github/fralfaro/MAT281/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAT281 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [5]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [29]:
#a
dfs = []
for archivo in archivos_anio:
    df_temp = pd.read_csv(archivo)
    df_temp.columns = df_temp.columns.str.lower()  #minúsculas
    dfs.append(df_temp)

df_anio = pd.concat(dfs, ignore_index=True)

print(df_anio.head())

#b
duplicados = df_codigos[df_codigos.duplicated("codigo_iso", keep=False)]
duplicados

df_codigos = df_codigos.drop(df_codigos[df_codigos["codigo_iso"] == "ZWE"].index)
print(duplicados)

#c
df= pd.merge(df_anio, df_codigos, on='codigo_iso')
print(df.head())

  codigo_iso  anio  indice  ranking
0        AFG  2001    35.5     59.0
1        AGO  2001    30.2     50.0
2        ALB  2001     NaN      NaN
3        AND  2001     NaN      NaN
4        ARE  2001     NaN      NaN
Empty DataFrame
Columns: [codigo_iso, pais]
Index: []
  codigo_iso  anio  indice  ranking                    pais
0        AFG  2001    35.5     59.0             Afghanistán
1        AGO  2001    30.2     50.0                  Angola
2        ALB  2001     NaN      NaN                 Albania
3        AND  2001     NaN      NaN                 Andorra
4        ARE  2001     NaN      NaN  Emiratos Árabes Unidos




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [75]:
#Estructura del DataFrame
print("ESTRUCTURA DEL DATAFRAME")
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])
print("Columnas:", df.columns[0],",", df.columns[1],",", df.columns[2],",", df.columns[3],",", df.columns[4])
print("Tipos de datos por columnas:\n", df.dtypes)
print("Todas las columnas tienen un tipo de dato adecuado")
print("------------------------------------------------------------------------------")

#Resumen estadístico
print("RESUMEN ESTADÍSTICO")
print(df.describe())
print("En ambas columnas el promedio es mucho más pequeño que la media, por lo que es posible deducir que la mayoría de datos están concentrados en el tercer cuartil, esto hace que la media no sea representativa")
print("En la columna indice, el mínimo es cero, el máximo es 64536 y el promedio es 206.739211 ")

print("------------------------------------------------------------------------------")
valor_max_i = df_anio["indice"].max()
pais_max_i = df_anio.loc[df_anio["indice"] == valor_max_i, "codigo_iso"]
print("País con máximo índice:\n", pais_max_i)
print("------------------------------------------------------------------------------")
valor_min_i = df_anio["indice"].min()
pais_min_i = df_anio.loc[df_anio["indice"] == valor_min_i, "codigo_iso"]
print("País con mínimo índice:\n", pais_min_i)
print("------------------------------------------------------------------------------")
valor_max_r = df_anio["ranking"].max()
pais_max_r = df_anio.loc[df_anio["ranking"] == valor_max_r, "codigo_iso"]
print("País con máximo ranking:\n", pais_max_r)
print("------------------------------------------------------------------------------")
valor_min_r = df_anio["ranking"].min()
pais_min_r = df_anio.loc[df_anio["ranking"] == valor_min_r, "codigo_iso"]
print("País con mínimo ranking:\n", pais_min_r)
print("------------------------------------------------------------------------------")

#Datos faltantes
print("DATOS FALTANTES")
print(df.isnull().sum())
print("------------------------------------------------------------------------------")
print("Proporción de valores faltantes en porcentaje:\n", df_anio.isnull().mean() *100)
print("Podemos ver en la tabla de arriba que no hay columnas con más del 30% ed datos faltantes")
print("------------------------------------------------------------------------------")

#Unicidad y duplicados
print("UNICIDAD Y DUPLICADOS")
print("Cantidad de paises únicos:", df['codigo_iso'].nunique())
print("Cantidad de años únicos:", df['anio'].nunique())
duplicadas = df_anio.duplicated()
print("Hay filas duplicadas:", duplicadas.any())
print("No hay incosistencias entre países, ya que anteriormente eliminamos los códigos iso duplicados(para países distintos) y solo ocurría con 'ZWE'")
print("------------------------------------------------------------------------------")

ESTRUCTURA DEL DATAFRAME
Cantidad de filas: 3043
Cantidad de columnas: 5
Columnas: codigo_iso , anio , indice , ranking , pais
Tipos de datos por columnas:
 codigo_iso     object
anio            int64
indice        float64
ranking       float64
pais           object
dtype: object
Todas las columnas tienen un tipo de dato adecuado
------------------------------------------------------------------------------
RESUMEN ESTADÍSTICO
              anio        indice        ranking
count  3043.000000   2648.000000    2820.000000
mean   2009.941176    206.739211     480.173404
std       5.786029   2703.631345    6494.364772
min    2001.000000      0.000000       1.000000
25%    2005.000000     15.250000      34.000000
50%    2009.000000     27.920000      70.000000
75%    2015.000000     41.000000     110.000000
max    2019.000000  64536.000000  121056.000000
En ambas columnas el promedio es mucho más pequeño que la media, por lo que es posible deducir que la mayoría de datos están concentrados




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [97]:
# respuesta
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
       'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
       'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
       'USA', 'VEN']

#a
df_america = df_anio[df_anio["codigo_iso"].isin(america)].copy()
resultados = []
años = sorted(df_america["anio"].unique())

for anio in años:
    df_año = df_america[df_america["anio"] == anio]
    pais_min_indice = df_año.nsmallest(1, "indice")["codigo_iso"].values[0]
    pais_max_indice = df_año.nlargest(1, "indice")["codigo_iso"].values[0]

    resultados.append({
        "anio": anio,
        "pais_mas_libre": pais_min_indice,
        "pais_menos_libre": pais_max_indice
    })

df_result_for = pd.DataFrame(resultados)
print(df_result_for.head())
print("------------------------------------------------------------------------------")


#b
df_america_clean = df_america.dropna(subset=["indice"])

min_indice = df_america_clean.loc[df_america_clean.groupby("anio")["indice"].idxmin(), ["anio", "codigo_iso", "indice"]]
min_indice = min_indice.rename(columns={"codigo_iso": "pais_mas_libre", "indice": "indice_min"})

max_indice = df_america_clean.loc[df_america_clean.groupby("anio")["indice"].idxmax(), ["anio", "codigo_iso", "indice"]]
max_indice = max_indice.rename(columns={"codigo_iso": "pais_menos_libre", "indice": "indice_max"})

# Unir ambos resultados por año
df_result_groupby = pd.merge(min_indice, max_indice, on="anio")
print(df_result_groupby.head())


   anio pais_mas_libre pais_menos_libre
0  2001            CAN              CUB
1  2002            TTO              CUB
2  2003            TTO              ARG
3  2004            TTO              CUB
4  2005            BOL              CUB
------------------------------------------------------------------------------
   anio pais_mas_libre  indice_min pais_menos_libre  indice_max
0  2001            CAN         0.8              CUB       90.30
1  2002            TTO         1.0              CUB       97.83
2  2003            TTO         2.0              ARG    35826.00
3  2004            TTO         2.0              CUB       87.00
4  2005            BOL         4.5              CUB       95.00


### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [109]:
tabla_pivot = pd.pivot_table(
    df_anio,
    index="codigo_iso",
    columns="anio",
    values="indice",
    aggfunc="max",
    fill_value=0
)
print(tabla_pivot.head())
print("------------------------------------------------------------------------------")
#a

pmax = tabla_pivot.max(axis=1).idxmax()
vmax = tabla_pivot.max(axis=1).max()

pmin = tabla_pivot[tabla_pivot>0].min(axis=1).idxmin()
vmin = tabla_pivot[tabla_pivot>0].min(axis=1).min()

print(f"Mayor índice: {vmax}, país: {pmax}")
print(f"Menor índice (distinto de cero): {vmin}, país: {pmin}")
print("------------------------------------------------------------------------------")
#b
promedio_anual = tabla_pivot.mean(axis=0)

anio_max_prom = promedio_anual.idxmax()
anio_min_prom = promedio_anual.idxmin()

print(f"Año con promedio más alto: {anio_max_prom}, promedio: {promedio_anual.max()}")
print(f"Año con promedio más bajo: {anio_min_prom}, promedio: {promedio_anual.min()}")
print("------------------------------------------------------------------------------")
#c
variabilidad = tabla_pivot.max(axis=1) - tabla_pivot.min(axis=1)
pais_mayor_var = variabilidad.idxmax()
valor_mayor_var = variabilidad.max()

print(f"País con mayor variabilidad: {pais_mayor_var}, diferencia: {valor_mayor_var}")
print("------------------------------------------------------------------------------")
#d
pais_constante = tabla_pivot[tabla_pivot.max(axis=1) == tabla_pivot.min(axis=1)].index.tolist()
print("Países con índice constante:", pais_constante)
print("No existen países con índices constantes")
print("------------------------------------------------------------------------------")
#e
pais_sin_datos = tabla_pivot[(tabla_pivot==0).all(axis=1)].index.tolist()
print("Países sin datos:", pais_sin_datos)
print("No existen países sin datos, ya que estos casos fueron quitados previamente")


anio        2001   2002   2003   2004   2005   2006   2007   2008   2009  \
codigo_iso                                                                 
AFG         35.5  40.17  28.25  39.17  44.25  56.50  59.25  54.25  51.67   
AGO         30.2  28.00  26.50  18.00  21.50  26.50  29.50  36.50  28.50   
ALB          0.0   6.50  11.50  14.17  18.00  25.50  16.00  21.75  21.50   
AND          0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   
ARE          0.0  37.00  50.25  25.75  17.50  20.25  14.50  21.50  23.75   

anio         2012   2013   2014   2015   2017   2018   2019  
codigo_iso                                                   
AFG         37.36  37.07  37.44  37.75  39.46  37.28  36.55  
AGO         37.80  36.50  37.84  39.89  40.42  38.35  34.96  
ALB         30.88  29.92  28.77  29.92  29.92  29.49  29.84  
AND          6.82   6.82  19.87  19.87  21.03  22.21  24.63  
ARE         33.49  36.03  36.73  36.73  39.39  40.86  43.63  
---------------------------------